# ccdcpp Benchmark

This notebook benchmarks **PyCCD** against **ccdcpp** and demonstrates single-pixel and image-cube processing.

In [1]:
import time
import numpy as np

import ccd
import ccdcpp

def read_data(path):
    """Load a sample file containing acquisition days and spectral values.

    The first column is assumed to be the day number, subsequent columns
    correspond to the day number. This improves readability of large datasets.

    Args:
        path: location of CSV containing test data

    Returns:
        A 2D numpy array.
    """
    return np.genfromtxt(path, delimiter=',', dtype=np.int64).T

In [ ]:
params={
    "QA_BITPACKED":False,
    "QA_FILL":255,
    "QA_CLEAR":0,
    "QA_WATER":1,
    "QA_SHADOW":2,
    "QA_SNOW":3,
    "QA_CLOUD":4,
}
dates, blues, greens, reds, nirs, swir1s, swir2s, thermals, qas = read_data("./ccd/procedure/test_3657_3610_observations.csv")

In [3]:
dates_cpp = np.asarray(dates.copy(),dtype=np.int64,order="C")
qas_cpp = np.asarray(qas.copy(),dtype=np.uint8,order="C")
spectra_cpp = np.asarray(np.stack((blues, greens, reds, nirs, swir1s, swir2s, thermals)), dtype=np.float64,order="C")

hoptions = ccdcpp.HarmonicOptions()
loptions = ccdcpp.LassoOptions()

hoptions.PEEK_SIZE = 6
hoptions.MEOW_SIZE = 12

loptions.max_iter = 1000
loptions.alpha = 1.0
loptions.tolerance = 1e-4
loptions.fit_intercept = True

In [4]:
N = 20
total = 0.0
for _ in range(N):
    s = time.perf_counter()
    py_results = ccd.detect(dates ,blues, greens, reds, nirs, swir1s, swir2s, thermals, qas, params=params)
    total += time.perf_counter() - s
py_time = total / N
print(f"PyCCD: {py_time:.4f} seconds")

PyCCD: 0.6359 seconds


In [5]:
total = 0.0
for _ in range(N):
    d = dates_cpp.copy()
    q = qas_cpp.copy()
    s = spectra_cpp.copy()
    b = time.perf_counter()
    cpp_results = ccdcpp.detect(d, s, q, hoptions, loptions)
    total += time.perf_counter() - b
cpp_time = total/N
print(f"ccdcpp: {cpp_time:.4f} seconds")
print(f"Speedup: {py_time / cpp_time:.1f}x")

ccdcpp: 0.0100 seconds
Speedup: 63.8x


## Optional Output Validation

In [6]:
assert(len(cpp_results['models']) == len(py_results['change_models']))
for m in range(len(cpp_results["models"])):
    print(f"\nModel {m}")

    # Integer/exact fields
    for a in [
        "start_day",
        "end_day",
        "break_day",
        "observation_count",
        "curve_qa",
    ]:
        print(f"{a:20s}:",
              cpp_results["models"][m][a] == py_results["change_models"][m][a])

    # Float field
    print(
        f"{'change_probability':20s}:",
        np.isclose(
            cpp_results["models"][m]["change_probability"],
            py_results["change_models"][m]["change_probability"],
            rtol=1e-8,
            atol=1e-10,
        ),
    )

    for i, b in enumerate(
        ["blue", "green", "red", "nir", "swir1", "swir2", "thermal"]
    ):
        cm = cpp_results["models"][m]["bands"][i]
        pm = py_results["change_models"][m][b]

        print(f"\n  {b}")

        print(
            "    rmse :",
            np.isclose(cm["rmse"], pm["rmse"], rtol=1e-6, atol=1e-8),
        )

        print(
            "    magn :",
            np.isclose(cm["magnitude"], pm["magnitude"], rtol=1e-6, atol=1e-8),
        )

        print(
            "    inter:",
            np.isclose(cm["intercept"], pm["intercept"], rtol=1e-6, atol=1e-8),
        )

        print(
            "    coeff:",
            np.allclose(
                cm["coefficients"],
                pm["coefficients"],
                rtol=1e-6,
                atol=1e-8,
            ),
        )


Model 0
start_day           : True
end_day             : True
break_day           : True
observation_count   : True
curve_qa            : True
change_probability  : True

  blue
    rmse : True
    magn : True
    inter: True
    coeff: True

  green
    rmse : True
    magn : True
    inter: True
    coeff: True

  red
    rmse : True
    magn : True
    inter: True
    coeff: True

  nir
    rmse : True
    magn : True
    inter: True
    coeff: True

  swir1
    rmse : True
    magn : True
    inter: True
    coeff: True

  swir2
    rmse : True
    magn : True
    inter: True
    coeff: True

  thermal
    rmse : True
    magn : True
    inter: True
    coeff: True

Model 1
start_day           : True
end_day             : True
break_day           : True
observation_count   : True
curve_qa            : True
change_probability  : True

  blue
    rmse : True
    magn : True
    inter: True
    coeff: True

  green
    rmse : True
    magn : True
    inter: True
    coeff: True

  re

## Image Cube Example

In [7]:
rows,cols=100,100
cube_dates = dates_cpp.copy()
cube_qas = np.broadcast_to(qas_cpp,(rows, cols, qas_cpp.shape[0])).copy()
cube_spectra = np.broadcast_to(spectra_cpp, (rows, cols, *spectra_cpp.shape)).copy()

start=time.perf_counter()
cube_results = ccdcpp.detect_cube(cube_dates, cube_spectra, cube_qas, hoptions, loptions)
elapsed=time.perf_counter()-start
print(f"{rows}x{cols} cube processed in {elapsed:.2f} seconds")
print(f"Pixels processed: {len(cube_results)}")

100x100 cube processed in 23.62 seconds
Pixels processed: 10000


## Benchmark Results

The following results were obtained using the benchmark dataset included in the original PyCCD repository (`test_3657_3610_observations.csv`).

| Implementation | Average Runtime (20 runs) |
|----------------|--------------------------:|
| PyCCD | 0.6405 s |
| ccdcpp | 0.0102 s |
| **Speedup** | **62.0×** |

> **Note:** This benchmark was performed on the sample dataset included with PyCCD and is intended to illustrate the performance improvements of the C++ implementation. Actual speedups will vary depending on hardware, compiler optimizations, dataset size, and workload. In addition to substantially reducing execution time, `ccdcpp` is designed to minimize memory allocations through reusable workspaces and zero-copy NumPy interoperability.